# Backtesting

Backtesting applies portfolio rules to historical or simulated data to estimate how a strategy would have behaved.

Abbreviations used in this notebook:

- **NAV**: Net Asset Value, the value path of a portfolio.
- **SMA**: Simple Moving Average.
- **bps**: Basis points, where 100 bps equals 1 percent.
- **CHF**: Swiss franc, used only as an illustrative currency.

## 1. Intuition

A backtest is a rehearsal, not proof. It helps identify how rules behave through time, including returns, volatility, drawdowns, turnover, and transaction costs.

The danger is overfitting: creating rules that look good on past data but do not generalize.

## 2. Mathematics

Portfolio return:

$$
R_{p,t} = \sum_i w_{i,t-1}R_{i,t}
$$

Transaction cost approximation:

$$
Cost_t = Turnover_t \times CostRate
$$

Turnover:

$$
Turnover_t = \sum_i |w_{i,t} - w_{i,t-1}|
$$

Net return:

$$
R_{net,t} = R_{gross,t} - Cost_t
$$

Where:
- $R_{p,t}$ = portfolio return at time $t$.
- $w_{i,t-1}$ = weight of asset $i$ set before period $t$ return is realized.
- $R_{i,t}$ = return of asset $i$ at time $t$.
- $Turnover_t$ = total absolute portfolio weight change at time $t$.
- $\text{CostRate}$ = transaction cost charged per unit of turnover.
- $R_net,t$ = return after transaction costs.


## 3. Implementation

We compare an equal-weight buy-and-hold portfolio with a monthly momentum-tilt strategy. Transaction costs are included in the strategy returns.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "03_portfolio_management" / "portfolio_utils.py"
spec = importlib.util.spec_from_file_location("portfolio_utils", helper_path)
portfolio_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(portfolio_utils)

plt.style.use("seaborn-v0_8-whitegrid")

returns = portfolio_utils.generate_synthetic_returns(periods=504, seed=321)
prices = portfolio_utils.returns_to_prices(returns)
assets = returns.columns.tolist()

rebalance_dates = returns.resample("ME").last().index
momentum = prices.pct_change(63)
transaction_cost_rate = 0.0010

weights_history = pd.DataFrame(index=returns.index, columns=assets, dtype=float)
previous_weights = np.repeat(1 / len(assets), len(assets))

for date in returns.index:
    if date in rebalance_dates and not momentum.loc[date].isna().any():
        ranks = momentum.loc[date].rank(ascending=False)
        signal = (len(assets) + 1 - ranks).to_numpy()
        target_weights = signal / signal.sum()
        previous_weights = target_weights
    weights_history.loc[date] = previous_weights

weights_history = weights_history.ffill().fillna(1 / len(assets))
strategy_gross = (weights_history.shift(1).fillna(1 / len(assets)) * returns).sum(axis=1)
turnover = weights_history.diff().abs().sum(axis=1).fillna(0)
strategy_net = strategy_gross - turnover * transaction_cost_rate
benchmark = returns.mean(axis=1)

backtest = pd.DataFrame({
    "benchmark": benchmark,
    "strategy_gross": strategy_gross,
    "strategy_net": strategy_net,
    "turnover": turnover,
})

backtest.head()

In [ ]:
summary = pd.DataFrame({
    "benchmark": {
        "annualized_return": portfolio_utils.annualized_return(backtest["benchmark"]),
        "annualized_volatility": portfolio_utils.annualized_volatility(backtest["benchmark"]),
    },
    "strategy_net": {
        "annualized_return": portfolio_utils.annualized_return(backtest["strategy_net"]),
        "annualized_volatility": portfolio_utils.annualized_volatility(backtest["strategy_net"]),
    },
})
summary.loc["sharpe"] = (summary.loc["annualized_return"] - 0.015) / summary.loc["annualized_volatility"]
summary.round(4)

## 4. Visualization

A backtest should show wealth curves, drawdowns, and allocation changes. Return-only charts hide too much.

In [ ]:
wealth = (1 + backtest[["benchmark", "strategy_net"]]).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
wealth.plot(ax=axes[0], color=["#9a6b2f", "#2f6f8f"])
axes[0].set_title("Backtest Wealth Curves")
axes[0].set_ylabel("Growth of 1")
axes[0].legend(["Equal-weight benchmark", "Momentum strategy net"])

portfolio_utils.drawdown(backtest["strategy_net"])["drawdown"].plot(ax=axes[1], color="#2f6f8f", label="Strategy")
portfolio_utils.drawdown(backtest["benchmark"])["drawdown"].plot(ax=axes[1], color="#9a6b2f", label="Benchmark")
axes[1].set_title("Drawdowns")
axes[1].set_ylabel("Drawdown")
axes[1].yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
weights_history.plot.area(figsize=(11, 4), alpha=0.8)
plt.title("Strategy Allocation Through Time")
plt.ylabel("Weight")
plt.gca().yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
plt.tight_layout()
plt.show()

## 5. Application

A backtest is useful when it includes costs, realistic rebalancing, and a benchmark. A strategy that wins before costs but loses after costs may not be implementable.

In [ ]:
cost_summary = pd.Series({
    "average_monthly_turnover": backtest.loc[backtest["turnover"] > 0, "turnover"].mean(),
    "total_turnover": backtest["turnover"].sum(),
    "total_cost_drag": (backtest["strategy_gross"] - backtest["strategy_net"]).sum(),
    "strategy_max_drawdown": portfolio_utils.drawdown(backtest["strategy_net"])["drawdown"].min(),
    "benchmark_max_drawdown": portfolio_utils.drawdown(backtest["benchmark"])["drawdown"].min(),
})

cost_summary.to_frame("value")

## 6. Reflection

- Backtests estimate behavior; they do not guarantee future returns.
- Costs, turnover, and rebalancing rules matter.
- Drawdowns often reveal more than annualized return.
- A strategy should be compared with a simple benchmark.

Questions to answer after running the notebook:

1. Did the strategy beat the benchmark after costs?
2. How much did transaction costs matter?
3. Did the strategy improve drawdown?
4. What could make this backtest unrealistic?